# 🤖 Model Context Protocol (MCP) Agent Workflows in `embpy`

This tutorial demonstrates how to leverage **`embpy`'s native Model Context Protocol (MCP) server** to expose bioinformatics and single-cell representation capabilities to AI assistants (Claude Desktop, Cursor, Windsurf, or custom autonomous agents).

It needs embpy with the MCP, model and scIB extras (`pip install "embpy[mcp,models,scib]"`), and network access for the identifier-mapping step (Ensembl / MyGene.info).

---

## 1. Introduction & Architecture

The **Model Context Protocol (MCP)** is an open standard that allows Large Language Models (LLMs) and agent frameworks to securely call external tools and retrieve domain context.

In `embpy`, the MCP server operates over standard input/output (`stdio`) using JSON-RPC 2.0 frames:

- **Strict Stdio Isolation:** Model logging, warnings, and library output (e.g., from PyTorch, Scanpy, scIB) are diverted to `sys.stderr` so that `sys.stdout` remains pure, uncorrupted JSON-RPC.
- **Resilient Error Traps:** Tool exceptions (such as out-of-memory or missing observation columns) return actionable JSON error payloads rather than crashing the process.
- **9 Bioinformatics Tools:**
  1. `inspect_data`: Profile dataset dimensions, layers, raw counts vs normalized expression, and candidate label/batch columns.
  2. `resolve_identifiers`: Map gene symbols <-> Ensembl IDs, and compound names or SMILES -> canonical SMILES.
  3. `preprocess_counts`: QC filtering, normalization, log1p and highly variable gene selection, saved to disk.
  4. `list_model_registry`: Report the models each embedding modality can run, the compute hardware (CUDA/MPS/CPU), and which optional extras are installed.
  5. `compute_embeddings`: Compute embeddings for one modality (cells, genes, proteins, molecules, ...) via `BioEmbedder`.
  6. `annotate_metadata`: Look up GO terms, pathways, domains, diseases, interactions or expression for genes, proteins, molecules and cell lines.
  7. `compute_umap`: Generate kNN graphs and 2D UMAP coordinates for visual inspection.
  8. `benchmark_scib`: Score biological conservation and batch integration with scIB.
  9. `benchmark_perturbations`: Benchmark predicted vs observed perturbation responses with ArcInstitute `cell-eval`.

This notebook calls six of them as Python functions, the same functions the server exposes; [tutorial 06](06_universal_mcp_agents.ipynb) runs all nine as one pipeline.


In [ ]:
import json
import os
import tempfile

import numpy as np
import pandas as pd
from anndata import AnnData
from IPython.display import Markdown, display

import embpy
from embpy.mcp import (
    benchmark_scib,
    compute_embeddings,
    compute_umap,
    create_server,
    inspect_data,
    list_model_registry,
    resolve_identifiers,
)

print(f"embpy loaded successfully from: {embpy.__file__}")


## 2. Programmatic Tool Inspection

AI assistants discover capabilities by requesting `tools/list` from the MCP server, which answers with the nine tools above. We can programmatically inspect the registered tools, their parameters, and documentation.


In [ ]:
# Instantiate the MCP server instance
server = create_server("embpy-tutorial-server")


def tool_schema(tool):
    """Return the input JSON schema of an MCP tool (`input_schema` in mcp 2.x, `inputSchema` in 1.x)."""
    return getattr(tool, "input_schema", None) or getattr(tool, "inputSchema", None)


async def inspect_server_tools():
    """Print the name, summary line and (truncated) input schema of every registered tool."""
    tool_list = await server.list_tools()
    print(f"Discovered {len(tool_list)} registered MCP tools:\n")
    for tool in tool_list:
        print(f"🔹 Tool: {tool.name}")
        doc_first_line = (tool.description or "").strip().split("\n")[0]
        print(f"   Description: {doc_first_line}")
        schema = tool_schema(tool)
        schema_str = json.dumps(schema, indent=2) if isinstance(schema, dict) else str(schema)
        print(f"   Schema: {schema_str[:160]}...\n")


await inspect_server_tools()


## 3. Hardware & Foundation Model Discovery

Agents first query `list_model_registry` to find the compute device, the models each modality can run, and which optional extras are installed (`optional_extras_status` has one `installed` flag per extra).


In [ ]:
# Query available models and hardware
env_info = list_model_registry()
print("Compute Device:", env_info["compute_device"], f"({env_info['device_details']})")
print("Runnable Single-Cell Models:", env_info["runnable_singlecell_models"])
print("Registered Single-Cell Models Count:", len(env_info["available_models"]["single_cell"]))

# Which optional extras are installed
extras_df = pd.DataFrame.from_dict(env_info["optional_extras_status"], orient="index")
extras_df[["installed"]]


## 4. Preparing a Sample Single-Cell Dataset

We create a self-contained AnnData dataset mimicking a single-cell experiment (cells with cell-type labels, batches, and perturbation conditions). The first eight genes carry real HGNC symbols and the rest are placeholders. The counts are random Poisson draws and the labels repeat in a fixed cycle, so the data carry no biological signal: the scores further down only show what the tools return.


In [ ]:
work_dir = tempfile.mkdtemp(prefix="embpy_mcp_demo_")
dataset_path = os.path.join(work_dir, "pbmc_demo.h5ad")

rng = np.random.RandomState(42)
n_cells, n_genes = 60, 50
counts = rng.poisson(lam=2.5, size=(n_cells, n_genes)).astype(np.float32)

adata = AnnData(X=counts)
adata.obs_names = [f"cell_{i}" for i in range(n_cells)]
# Real gene symbols (used to test identifier mapping), then placeholder names
real_symbols = ["TP53", "GAPDH", "CD4", "CD8A", "MS4A1", "IL7R", "GNLY", "NKG7"]
adata.var_names = real_symbols + [f"GENE_{i}" for i in range(len(real_symbols), n_genes)]

adata.obs["cell_type"] = (["T_cell", "B_cell", "NK_cell"] * (n_cells // 3))[:n_cells]
adata.obs["batch"] = (["batch_A", "batch_B"] * (n_cells // 2))[:n_cells]
adata.obs["perturbation"] = (["ctrl", "ctrl", "stim_IFN", "stim_IFN"] * 20)[:n_cells]
adata.obs["split"] = (["train", "test"] * 30)[:n_cells]
adata.layers["counts"] = counts.copy()

adata.write_h5ad(dataset_path)
print(f"Sample dataset saved to: {dataset_path} ({n_cells} cells, {n_genes} genes)")


## 5. Simulated Agent Workflow

### Step 1: Dataset Inspection & Profiling
The agent inspects the file using `inspect_data` to verify cell/gene counts, categorical columns, and expression nature (raw counts vs log-normalized).


In [ ]:
inspection = inspect_data(dataset_path)
print("Status:", inspection["status"])
print("Cell Count (n_obs):", inspection["n_obs"])
print("Gene Count (n_vars):", inspection["n_vars"])
print("Detected Expression Nature:", inspection["expression_nature"]["detected_nature"])
print("Observation Columns:", inspection["obs_columns"])
print("Layers:", inspection["layers"])


### Step 2: Gene Identifier Harmonization
Single-cell foundation models use different gene vocabularies: Geneformer is built on Ensembl IDs, scGPT on gene symbols. The agent converts symbols to Ensembl IDs using `resolve_identifiers`.


In [ ]:
# Map sample gene symbols to Ensembl IDs
mapping = resolve_identifiers(
    identifiers=real_symbols[:5],
    target_id_type="ensembl",
    organism="human",
)
print("Status:", mapping["status"])
print("Mapped:", f"{mapping['mapped_count']} of {mapping['total_submitted']} symbols")
print("Mapped Gene Sample:")
for sym, ens in mapping["sample_mapping"].items():
    print(f"  {sym} -> {ens}")


### Step 3: Embedding Generation
The agent computes cell embeddings using `compute_embeddings` (modality `single_cell`). Here we use PCA. With the default `preprocessing="auto"` the raw counts are normalized and log-transformed before PCA (QC thresholds are scaled down for this small dataset), and the updated dataset is saved with the embeddings.


In [ ]:
embedded_dataset = os.path.join(work_dir, "pbmc_embedded.h5ad")

embed_result = compute_embeddings(
    path=dataset_path,
    models=["pca"],
    entity_type="single_cell",
    output_path=embedded_dataset,
    device="cpu",
)
print("Embed Status:", embed_result.get("status"))
if embed_result.get("status") == "success":
    print("Computed Models:", embed_result["models_computed"])
    print("Embedding Keys in .obsm:", embed_result["embedding_keys"])
    print("Shapes:", embed_result["embedding_shapes"])
else:
    print("Embedding Diagnostics:", embed_result.get("error"))
    print("Suggestion:", embed_result.get("suggestion", "Ensure required dependencies are installed."))


### Step 4: 2D UMAP Coordinates
The agent constructs a kNN neighborhood graph and calculates 2D UMAP coordinates using `compute_umap`.


In [ ]:
umap_dataset = os.path.join(work_dir, "pbmc_umap.h5ad")

umap_result = compute_umap(
    path=embedded_dataset,
    embedding_key="X_pca",
    output_path=umap_dataset,
    n_neighbors=10,
)
print("UMAP Status:", umap_result.get("status"))
if umap_result.get("status") == "success":
    print("UMAP Key in .obsm:", umap_result["umap_key"])
    print("Coordinate Array Shape:", umap_result["umap_shape"])
else:
    print("UMAP Diagnostics:", umap_result.get("error"))
    print("Suggestion:", umap_result.get("suggestion"))


### Step 5: scIB Quality Benchmarking
The agent evaluates biological conservation and batch integration using `benchmark_scib`. The labels of this toy dataset are unrelated to its random counts, so expect low bio-conservation scores. A metric that cannot run in the current environment (for example kBET without `rpy2`) is reported as NaN and left out of the aggregate scores.


In [ ]:
scib_report_csv = os.path.join(work_dir, "scib_scores.csv")
target_dataset = umap_dataset if os.path.exists(umap_dataset) else embedded_dataset

scib_result = benchmark_scib(
    path=target_dataset,
    embedding_keys=["X_pca"],
    label_key="cell_type",
    batch_key="batch",
    output_csv=scib_report_csv,
)

print("scIB Evaluation Status:", scib_result.get("status"))
if scib_result.get("status") == "success":
    print("Total Composite Score:", scib_result["metrics"]["X_pca"].get("total"))
    display(Markdown(scib_result["markdown_table"]))
else:
    print("scIB Diagnostics:", scib_result.get("error"))
    print("Suggestion:", scib_result.get("suggestion"))


## 6. Lightweight Agent Tool-Calling Loop

Here is how an LLM agent dispatcher coordinates tools using a standard tool-dispatch pattern. The registry maps MCP tool names to the embpy functions behind them:


In [ ]:
TOOL_REGISTRY = {
    "inspect_data": inspect_data,
    "resolve_identifiers": resolve_identifiers,
    "list_model_registry": list_model_registry,
    "compute_embeddings": compute_embeddings,
    "compute_umap": compute_umap,
    "benchmark_scib": benchmark_scib,
}


def execute_agent_tool_call(tool_name: str, arguments: dict):
    """Simulate autonomous agent tool execution with structured error traps."""
    if tool_name not in TOOL_REGISTRY:
        return {"status": "error", "error": f"Unknown tool: {tool_name}"}
    tool_func = TOOL_REGISTRY[tool_name]
    try:
        return tool_func(**arguments)
    except Exception as exc:  # noqa: BLE001 -- the agent gets the error back instead of a crashed loop
        return {"status": "error", "error": str(exc)}


# Example simulated tool execution
simulated_query = {"path": dataset_path}
response = execute_agent_tool_call("inspect_data", simulated_query)
print("Agent Received Response:")
print(f"  Status: {response['status']}, Cells: {response['n_obs']}, Genes: {response['n_vars']}")


## 7. Connect to an LLM Client (Optional: Claude & Ollama)

In real-world agentic systems, you connect the `embpy` MCP tool suite to an LLM provider such as **Anthropic Claude** or a locally hosted model via **Ollama**. MCP clients such as Claude Code or Claude Desktop start the server themselves (`python -m embpy.mcp.server`; see `docs/mcp/client-setup.md`); the cells below wire the same tools into a chat API by hand.

The LLM receives the schema of each bioinformatics tool, taken from the MCP server's own tool list so that names and parameters match the server, reasons about user inquiries, and decides when to emit a structured tool call. The application executes the tool via `embpy` and returns the output back to the LLM for a final grounded scientific response. Without an `ANTHROPIC_API_KEY` or a running Ollama server, the cells print setup instructions instead.


In [ ]:
# =========================================================================
# Option A: End-to-End Tool Calling with Anthropic Claude
# =========================================================================
# Tool definitions come from the MCP server's tool list, so they always match the server.
server_tools = await server.list_tools()
mcp_tools = {tool.name: tool for tool in server_tools}
claude_tools_schema = [
    {
        "name": name,
        "description": mcp_tools[name].description,
        "input_schema": tool_schema(mcp_tools[name]),
    }
    for name in ("inspect_data", "resolve_identifiers")
]

if os.environ.get("ANTHROPIC_API_KEY"):
    try:
        import anthropic

        client = anthropic.Anthropic()

        user_prompt = f"Please inspect the single-cell dataset at '{dataset_path}' and tell me how many cells and genes it contains."
        print(f"User Prompt: {user_prompt}\n")
        messages = [{"role": "user", "content": user_prompt}]

        # Agent loop: call Claude, run the tools it asks for, send the results back, until it answers.
        while True:
            response = client.beta.messages.create(
                model="claude-opus-5-5",
                max_tokens=16000,
                # If a safety classifier declines the request, the API retries it on a fallback model.
                betas=["server-side-fallback-2026-07-01"],
                fallbacks="default",
                tools=claude_tools_schema,
                messages=messages,
            )
            if response.stop_reason != "tool_use":
                break
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"🤖 Claude requested tool: '{block.name}'")
                    print(f"   Arguments: {block.input}")
                    tool_result = execute_agent_tool_call(block.name, block.input)
                    tool_results.append(
                        {
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": json.dumps(tool_result, default=str),
                            "is_error": tool_result.get("status") == "error",
                        }
                    )
            # Every tool result of one turn goes back in a single message.
            messages.append({"role": "user", "content": tool_results})

        final_text = "".join(block.text for block in response.content if block.type == "text")
        print(f"\n🤖 Claude Final Response (stop_reason={response.stop_reason}):\n{final_text}")
    except Exception as err:  # noqa: BLE001 -- optional demo: report the problem and keep the notebook running
        print(f"Claude API execution encountered an error: {err}")
else:
    print("ℹ️ ANTHROPIC_API_KEY is not set.")
    print("   To run live Claude tool-calling:")
    print("   1. pip install anthropic")
    print("   2. export ANTHROPIC_API_KEY='sk-ant-api03-...'")
    print("   3. Re-run this cell to see Claude autonomously inspect the dataset.")


In [ ]:
# =========================================================================
# Option B: End-to-End Tool Calling with Local Ollama (e.g. Llama 3.1)
# =========================================================================

ollama_tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "inspect_data",
            "description": mcp_tools["inspect_data"].description,
            "parameters": tool_schema(mcp_tools["inspect_data"]),
        },
    }
]

try:
    import ollama

    user_prompt = f"Inspect the single-cell dataset at '{dataset_path}' and report its observation columns."
    print(f"User Prompt: {user_prompt}\n")

    # Step 1: Send prompt and tool definitions to local Ollama model
    ollama_response = ollama.chat(
        model="llama3.1",
        messages=[{"role": "user", "content": user_prompt}],
        tools=ollama_tools_schema,
    )

    # Step 2: Check if model emitted tool calls
    if ollama_response.message.tool_calls:
        for tool in ollama_response.message.tool_calls:
            print(f"🦙 Ollama requested tool: {tool.function.name}")
            print(f"   Arguments: {tool.function.arguments}")
            tool_output = execute_agent_tool_call(tool.function.name, tool.function.arguments)

            # Step 3: Send tool result back to Ollama
            final_response = ollama.chat(
                model="llama3.1",
                messages=[
                    {"role": "user", "content": user_prompt},
                    ollama_response.message,
                    {
                        "role": "tool",
                        "content": json.dumps(tool_output, default=str),
                    },
                ],
            )
            print(f"\n🦙 Ollama Final Response:\n{final_response.message.content}")
    else:
        print(f"🦙 Ollama Direct Response:\n{ollama_response.message.content}")

except Exception as err:  # noqa: BLE001 -- optional demo: no package or no server only skips it
    print(f"ℹ️ Ollama connection skipped ({err}).")
    print("   To run local tool-calling with open-weights LLMs:")
    print("   1. Install Ollama: https://ollama.com")
    print("   2. Run: ollama run llama3.1")
    print("   3. pip install ollama")
    print("   4. Re-run this cell for zero-cost, local offline agent tool dispatching.")


In [ ]:
# Clean up temporary demonstration files
import shutil

shutil.rmtree(work_dir, ignore_errors=True)
print("Demonstration artifacts cleaned up successfully.")
